# Figure 1 (temporary): vulnerable-cluster contours in the $(z,\bar\phi)$ plane

Four combinations of degree and threshold distributions:
(a) zeta degrees and Pareto thresholds,
(b) zeta degrees and truncated Gaussian thresholds,
(c) Poisson degrees and Pareto thresholds,
(d) Poisson degrees and truncated Gaussian thresholds.

Contours mark $\kappa=1$, separating supercritical from subcritical vulnerable branching.
The vertical lines in the upper panels mark $\gamma_d=2.5$ and the percolation threshold $\gamma_d^*\approx3.479$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import poisson, norm
from scipy.special import zeta
from scipy.optimize import brentq
from figstyle import PALETTE, LW, REF_LW, FIGW_DOUBLE, save

kmax = 3000                                 # degree cutoff for the cascade sums
k = np.arange(1, kmax + 1)
GAMMA_TH = [2.05, 2.10, 2.25, 3.00]         # Pareto threshold exponents
SIGMA = 0.1                                 # truncated-Gaussian scale


In [2]:
# Threshold CDFs evaluated at a single-neighbour fraction 1/k.
def pareto_F(gth, phi_bar, kk):
    """Pareto threshold CDF with mean phi_bar, evaluated at 1/k."""
    xmin = phi_bar * (gth - 2.0) / (gth - 1.0)
    arg = kk * xmin
    return np.where(arg < 1.0, 1.0 - arg ** (gth - 1.0), 0.0)

# The [0,1]-truncated Gaussian mean is monotone in its location mu, so invert
# by interpolation (a root finder is unstable in the saturated tails).
_mu = np.linspace(-0.3, 1.3, 20001)
_mean = _mu + SIGMA * (norm.pdf(-_mu / SIGMA) - norm.pdf((1 - _mu) / SIGMA)) / (
    norm.cdf((1 - _mu) / SIGMA) - norm.cdf(-_mu / SIGMA))

def gaussian_mu(phi_bar):
    """Gaussian location whose [0,1]-truncated mean equals phi_bar."""
    return np.interp(phi_bar, _mean, _mu)

def gaussian_F(mu, kk, sigma=SIGMA):
    """Truncated and normalised Gaussian threshold CDF, evaluated at 1/k."""
    lo, hi = norm.cdf(-mu / sigma), norm.cdf((1 - mu) / sigma)
    return (norm.cdf((1.0 / kk - mu) / sigma) - lo) / (hi - lo)


In [3]:
# Degree distributions: Poisson p_k = e^-z z^k/k! and zeta p_k = k^-g/zeta(g).
def zeta_pmf(g):
    return k ** (-g) / zeta(g)

def zeta_mean(g):
    return zeta(g - 1) / zeta(g)

# Vulnerability matrices B[k, phi] = F(1/k), shared by the zeta and Poisson panels.
phi = np.linspace(0.05, 0.95, 200)
B_pareto = {gth: pareto_F(gth, phi[None, :], k[:, None]) for gth in GAMMA_TH}
mu = np.array([gaussian_mu(p) for p in phi])
B_gauss = gaussian_F(mu[None, :], k[:, None])

# Branching weight w_k = k(k-1) p_k.
gs = np.linspace(2.02, 8.0, 240)
z_zeta = zeta_mean(gs)
A_zeta = k * (k - 1) * zeta_pmf(gs[:, None])

zs = np.linspace(1.0, 8.0, 240)
A_poi = k * (k - 1) * poisson.pmf(k[:, None], zs[None, :]).T

# Gaussian tail: F(1/k) -> f_G(0)/k, so sum_k w_k F beyond kmax -> f_G(0) zeta(g-1,kmax+1)/zeta(g).
f0 = norm.pdf(-mu / SIGMA) / (SIGMA * (norm.cdf((1 - mu) / SIGMA) - norm.cdf(-mu / SIGMA)))
tail_zeta = np.outer(zeta(gs - 1, kmax + 1) / zeta(gs), f0)


In [4]:
# kappa = (1/z) sum_k k(k-1) p_k F(1/k).
Kappa = {
    ('zeta', 'pareto'): {gth: A_zeta @ B_pareto[gth] / z_zeta[:, None] for gth in GAMMA_TH},
    ('zeta', 'gauss'): (A_zeta @ B_gauss + tail_zeta) / z_zeta[:, None],
    ('poisson', 'pareto'): {gth: A_poi @ B_pareto[gth] / zs[:, None] for gth in GAMMA_TH},
    ('poisson', 'gauss'): A_poi @ B_gauss / zs[:, None],
}

# z(gamma) is strongly non-uniform in gamma, so resample kappa onto a uniform
# mean-degree grid before contouring (otherwise the contours distort).
z_uni = np.linspace(1.0, 8.0, 400)

def zeta_map(Km):
    """Resample a (gamma, phi) kappa map onto the uniform z grid, contour-ready."""
    return np.array([np.interp(z_uni, z_zeta[::-1], Km[::-1, j]) for j in range(Km.shape[1])])

# Reference zeta exponents: gamma_d = 2.5 and the percolation threshold gamma_d*.
gd_star = brentq(lambda g: zeta(g - 2) - 2 * zeta(g - 1), 3.01, 6.0)
x_ref = [zeta_mean(2.5), zeta_mean(gd_star)]

panels = [('(a)', 'zeta', 'pareto'), ('(b)', 'zeta', 'gauss'),
          ('(c)', 'poisson', 'pareto'), ('(d)', 'poisson', 'gauss')]
gth_color = dict(zip(GAMMA_TH, PALETTE))

fig, axs = plt.subplots(2, 2, figsize=(FIGW_DOUBLE, 5.5), constrained_layout=True)
for i, (ax, (lab, topo, thr)) in enumerate(zip(axs.flat, panels)):
    is_zeta = topo == 'zeta'
    Ks = Kappa[(topo, thr)]
    x, prep = (z_uni, zeta_map) if is_zeta else (zs, np.transpose)

    if thr == 'pareto':
        for gth in GAMMA_TH:
            ax.contour(x, phi, prep(Ks[gth]), levels=[1.0],
                       colors=[gth_color[gth]], linewidths=LW)
        handles = [Line2D([0], [0], color=gth_color[g], lw=LW, label=fr'$\gamma_{{\rm th}}={g:g}$')
                   for g in GAMMA_TH]
    else:
        ax.contour(x, phi, prep(Ks), levels=[1.0], colors=['black'], linewidths=LW)
        handles = [Line2D([0], [0], color='black', lw=LW, label=r'$\kappa=1$')]

    if is_zeta:
        for xl in x_ref:
            ax.axvline(xl, color='0.35', lw=REF_LW, ls='--')

    ax.set_xlim(1, 8)
    ax.set_ylim(0.05, 0.95)
    ax.text(0.04, 0.94, lab, transform=ax.transAxes, va='top', fontweight='bold')
    if i >= 2:
        ax.set_xlabel(r'$z$')
    if i % 2 == 0:
        ax.set_ylabel(r'$\bar\phi$')
    ax.legend(handles=handles, frameon=False, loc='upper right')

save(fig, "figure1_temp")
plt.show()


/var/folders/l3/dgm693wx7m3b1107vjtfb8k40000gq/T/ipykernel_2391/1937908500.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# checks: zeta normalisation and mean (tail-closed), gamma_d*, truncation means, kappa
g = 2.5
p_sum = (k ** (-g)).sum() + zeta(g, kmax + 1)
mean = (k * zeta_pmf(g)).sum() + zeta(g - 1, kmax + 1) / zeta(g)
assert np.isclose(p_sum, zeta(g), rtol=1e-12)
assert np.isclose(mean, zeta_mean(g), rtol=1e-9)
print(f"zeta(g=2.5): sum p_k = {p_sum:.12f}, mean = {mean:.9f} (analytic {zeta_mean(2.5):.9f})")

print(f"gamma_d* = {gd_star:.4f} (paper: 3.479); z(gamma_d*) = {zeta_mean(gd_star):.4f}")

for p in [0.2, 0.5, 0.8]:
    m_ = gaussian_mu(p)
    mean_ = m_ + SIGMA * (norm.pdf(-m_ / SIGMA) - norm.pdf((1 - m_) / SIGMA)) / (
        norm.cdf((1 - m_) / SIGMA) - norm.cdf(-m_ / SIGMA))
    assert abs(mean_ - p) < 1e-4  # interpolation accuracy
    print(f"truncated Gaussian: phi_bar={p} -> mu={m_:.5f} (mean {mean_:.6f})")

for gth in GAMMA_TH:
    B = B_pareto[gth]
    assert ((B >= 0) & (B <= 1)).all()
assert ((B_gauss >= 0) & (B_gauss <= 1)).all()

zz = Kappa[('poisson', 'gauss')]
assert np.isfinite(zz).all() and (zz > 0).all()

# On Poisson graphs kappa -> f_G(0) as z -> infinity.
z_big = 1000.0
K = (k * (k - 1) * poisson.pmf(k, z_big)) @ B_gauss / z_big
j = np.argmin(np.abs(phi - 0.05))
assert abs(K[j] - f0[j]) / f0[j] < 0.01
print(f"Poisson+Gaussian, phi=0.05: kappa(z=1e3)={K[j]:.4f}, f_G(0)={f0[j]:.4f}")
print("figure 1 checks passed")


zeta(g=2.5): sum p_k = 1.341487257251, mean = 1.947372466 (analytic 1.947372466)
gamma_d* = 3.4788 (paper: 3.479); z(gamma_d*) = 1.1954
truncated Gaussian: phi_bar=0.2 -> mu=0.19373 (mean 0.200000)
truncated Gaussian: phi_bar=0.5 -> mu=0.50000 (mean 0.500000)
truncated Gaussian: phi_bar=0.8 -> mu=0.80627 (mean 0.800000)
Poisson+Gaussian, phi=0.05: kappa(z=1e3)=16.2031, f_G(0)=16.3115
figure 1 checks passed
